# Run an active-learning sampler

One sampler, one dataset, one seed, one config — the full budget sweep for
exactly that. The notebook ends in a single zip whose name states the whole
configuration, so an ablation is a second run, not a second entry in a list.

On a Kaggle **T4 x2** session the budget list is split across both GPUs, one
worker process per card. Set `PARALLEL = False` to force serial.

Per budget this writes selected indices with their per-step acquisition scores,
the probe weights, the test predictions, the metrics table, a
sanity report and a run log — see `main.py`. The last cell zips all of it.

Re-running the notebook skips a run whose `_results.pt` already exists, so a
session that hits the 12-hour limit can simply be run again.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
# ---- EDIT THIS CELL ----
DATASET = "pathmnist"

# Any name in sampling/specs.py. ONE sampler per notebook run:
#   random | coreset | typiclust | activeft | tcm
#   margin | entropy | badge | dropquery | uncertainty_herding | refine
#   scalpel   <- this project's method
SAMPLER = "scalpel"
SEED = 42             # ONE seed. Sweeping seeds = re-running the notebook.

# Axes worth sweeping per sampler are in
# config/config.yaml; a few examples:
#   scalpel : {"uncertainty_mode": "visual_margin"}  ablation: plain UHerding weight
#             {"cell_pooling": "rff"}                kernel mean embedding (KDE)
#             {"missing_impute": "zero"}             patches with no nucleus
#             {"consistency_weight": 0.1}            couples the two probes
#   typiclust : {"k_nn": 10}
#   tcm       : {"transition_class_multiple": 2}
#   activeft  : {"temperature": 0.1}
#   dropquery : {"dropout_ratio": 0.5}
# Sweeping a baseline's axes is baseline TUNING: either do it for every method
# or none, and say which in the report.
# ONE config per run. `{}` uses config.yaml as-is; the visual_margin ablation
# is a SECOND run with OVERRIDES = {"uncertainty_mode": "visual_margin"}, which
# gets its own run_name and its own zip.
OVERRIDES = {}

# Leave RUN_NAME None so each variant gets a collision-safe name of its own.
RUN_NAME = None

# Use both T4s when the session has them.
PARALLEL = True

# Split this run's budget list across both GPUs (non-prefix-exact samplers only).
SPLIT_BUDGETS = True

# Starting points only; the next cell searches for the real directories.
FEATURE_DIR = "/kaggle/input/datasets/cryandrrich/nckh2026/features"
CELLVIT_DIR = "/kaggle/input/datasets/cryandrrich/nckh2026/cellvit_features"
OUTPUT_DIR = "/kaggle/working/checkpoints"

# Where a .npz dataset is re-exported as memory-mappable .npy files.
#
# An AL run reads no pixels -- it opens the dataset for labels, sample IDs and
# the cache fingerprint -- but NPZDataset reads a .npz EAGERLY, and
# PathMNIST-224 is ~15 GiB of uint8. Two GPU workers each holding their own
# copy is ~30 GiB, which is the whole Kaggle session, so one is killed before
# the sampler starts. Mapping costs scratch disk once and nothing per worker.
# Ignored for ImageFolder datasets (histoset, skintissue), which read per file.
MMAP_CACHE_DIR = "/kaggle/working/npz_mmap"

In [ ]:
from huggingface_hub import snapshot_download

print("Downloading facebook/dinov2-base ...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import yaml
import torch

import main
from sampling.specs import spec_for
from utils.parallel import run_variants_parallel, visible_gpu_count
from utils.kaggle import find_cellvit_cache, find_data_root, find_visual_cache
from utils.progress import format_duration

In [ ]:
DATA_ROOT = find_data_root()

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
print("data root:", DATA_ROOT)

In [ ]:
with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

dataset_info = config["datasets"][DATASET]
training_cfg = config["training"]
sampler_cfg = {**config.get("samplers", {}).get(SAMPLER, {}), **OVERRIDES}
spec = spec_for(SAMPLER)

data_path = Path(DATA_PATHS[DATASET])
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert torch.cuda.is_available(), "Attach a Kaggle GPU before running AL"
# One configuration per run: a list here would produce several results under
# one archive name, which could then not say which result was which.
assert isinstance(SEED, int), "SEED is one seed, not a list -- re-run the notebook to sweep"
assert isinstance(OVERRIDES, dict), "OVERRIDES is one config dict, not a list of variants"

# Only samplers that declare a cell view need the CellViT cache. Each seed has
# its own cache, because the train/test split is seeded.
if "cell_embeddings" in spec.needs:
    found = find_cellvit_cache(DATASET, SEED, hint=CELLVIT_DIR)
    assert found is not None, (
        f"No CellViT cache for {DATASET}_seed{SEED} under {CELLVIT_DIR} or "
        f"the default Kaggle input roots. Attach the extraction dataset."
    )
    CELLVIT_DIR = str(found)
    print("cellvit cache:", CELLVIT_DIR)

# The DINOv2 cache is optional: main.py re-extracts on a miss. It must not
# re-extract into a read-only /kaggle/input, which would only fail AFTER the
# whole forward pass, so fall back to a writable directory instead.
vit_name = config.get("models", {}).get("vit", "facebook/dinov2-base")
found = find_visual_cache(DATASET, SEED, vit_name, hint=FEATURE_DIR)
if found is not None:
    FEATURE_DIR = str(found)
    print("features cache:", FEATURE_DIR)
else:
    print(f"[features] not found for {DATASET}/seed{SEED}/{vit_name} - will extract this session.")
    print("  (a .npy with no matching manifest is rejected too -- row alignment")
    print("  cannot be verified without it.)")
    FEATURE_DIR = "/kaggle/working/features"

if not str(FEATURE_DIR).startswith("/kaggle/input"):
    Path(FEATURE_DIR).mkdir(parents=True, exist_ok=True)
SAVE_DIR = Path(OUTPUT_DIR) / DATASET
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"{SAMPLER}: {spec.passes} pass, prefix_exact={spec.prefix_exact} - {spec.why}")
print(f"GPUs visible: {visible_gpu_count()}")
print("config:", sampler_cfg)

In [ ]:
import time

BUDGETS = config["cumulative_budget"]
workers = visible_gpu_count() if PARALLEL else 1

# Budget sharding only helps a sampler whose budgets are independent runs.
# `spec.prefix_exact` is exactly that property, so it -- not a hand-kept list
# of names -- decides.
shard_budgets = SPLIT_BUDGETS and workers > 1 and not spec.prefix_exact and len(BUDGETS) > 1
if SPLIT_BUDGETS and spec.prefix_exact:
    print(f"[shard] {SAMPLER} is prefix-exact: one shared selection pass covers every")
    print("        budget, so its sweep stays on a single GPU.")

def budget_shards(budgets, n):
    """Round-robin, not contiguous: cost grows with the budget, so a contiguous
    split would hand one worker every large budget."""
    groups = [budgets[i::n] for i in range(n)]
    return [g for g in groups if g]

RUN = RUN_NAME or main._default_run_name(SAMPLER, sampler_cfg)
if SEED != config.get("random_seed", 42):
    RUN = f"{RUN}_s{SEED}"

base_kwargs = dict(
    data_path=str(data_path),
    sampler_name=SAMPLER,
    num_classes=dataset_info["num_classes"],
    data_descriptions=dataset_info.get("descriptions", {}),
    prompt_templates=config.get("prompt_templates", []),
    sampler_cfg=sampler_cfg,
    probe_epochs=training_cfg["probe_epochs"],
    probe_lr=training_cfg["probe_lr"],
    random_seed=SEED,
    save_dir=str(SAVE_DIR),
    verbose=True,
    model_cfg=config.get("models", {}),
    feature_cache_dir=FEATURE_DIR,
    mmap_cache_dir=MMAP_CACHE_DIR,
    cellvit_cache_dir=CELLVIT_DIR,
    run_name=RUN,
    device_string="cuda:0",
)

jobs = []
shard_tags = []
if (SAVE_DIR / f"{RUN}_results.pt").is_file():
    print(f"already finished, nothing to do: {RUN}_results.pt")
elif shard_budgets:
    shards = budget_shards(BUDGETS, workers)
    shard_tags = [f"shard{i}" for i in range(len(shards))]
    for tag, budgets in zip(shard_tags, shards):
        jobs.append((f"{RUN}:{tag}", dict(
            base_kwargs, cumulative_budget=budgets, shard_tag=tag,
        )))
else:
    jobs.append((RUN, dict(base_kwargs, cumulative_budget=BUDGETS)))

print(f"run: {RUN}")
for label, kwargs in jobs:
    print(f"   {label:40} budgets={kwargs['cumulative_budget']}")

# Export the .npz to memory-mappable .npy files ONCE, here in the parent.
#
# Must be the parent: two GPU workers exporting the same ~15 GiB concurrently
# would race on the same filenames. Once exported, each worker maps the same
# pages and the OS page cache serves them all from one copy, so a second worker
# costs no additional pixel memory.
#
# ImageFolder datasets skip this: they hold a path list and read per file, so
# they were never the problem.
if str(data_path).endswith(".npz"):
    from data.npz_mmap import export_npz_to_npy

    export_npz_to_npy(str(data_path), MMAP_CACHE_DIR)
    print(f"mmap export ready: {MMAP_CACHE_DIR}")
else:
    print("ImageFolder dataset: reads per file, no mmap export needed")

started = time.time()
results = run_variants_parallel(jobs, main.run_on_worker, num_workers=workers)

print("=" * 70)
for result in results:
    status = "ok" if result["ok"] else "FAILED"
    print(f"{result['label']:40} {status:8} {format_duration(result['seconds'])}")
failed = [r["label"] for r in results if not r["ok"]]
if results:
    print(f"total {format_duration(time.time() - started)} | "
          f"{len(results) - len(failed)}/{len(results)} succeeded")
assert not failed, f"jobs failed: {failed}"

if shard_tags:
    linear = main.merge_budget_shards(str(SAVE_DIR), RUN, shard_tags)
    print(f"[merge] {RUN}: {len(linear)} budgets -> {RUN}_results.pt")

In [ ]:
# Results table, in budget order.
#
# The dispatch cell above runs two GPUs in one output stream, so their progress
# lines interleave and a budget's numbers can land anywhere. This cell ignores
# all of that and reads the saved `<run>_results.pt`, which is the authoritative
# record -- so the table below is correctly ordered no matter how the logs came
# out, and re-running this cell alone reprints it without recomputing anything.
#
# Metrics only. Per-step acquisition scores, sigma and the sanity report live in
# the saved files for later analysis; printing them here would recreate exactly
# the wall of text this cell exists to avoid.
import torch

results_path = SAVE_DIR / f"{RUN}_results.pt"
assert results_path.is_file(), (
    f"no results at {results_path} -- the run above did not finish"
)
payload = torch.load(results_path, weights_only=False)
linear = payload["linear"]

print(f"{payload['sampler']}  |  {payload['dataset']}  |  seed {payload['seed']}")
print(f"run_name: {payload['run_name']}")
if payload.get("sharded_over"):
    print(f"budget shards: {', '.join(payload['sharded_over'])}")
print()

header = f"{'budget':>8}  {'accuracy':>9}  {'precision':>9}  {'recall':>9}  {'macro F1':>9}  {'select s':>9}"
print(header)
print("-" * len(header))
for budget in sorted(linear):
    row = linear[budget]
    print(f"{budget:>8}  {row['acc']:>9.4f}  {row['precision']:>9.4f}  "
          f"{row['recall']:>9.4f}  {row['f1']:>9.4f}  {row['selection_seconds']:>9.1f}")
print("-" * len(header))

best = max(linear, key=lambda b: linear[b]["acc"])
print(f"best accuracy {linear[best]['acc']:.4f} at budget {best}")

# A sanity severity worse than "ok" means the selection itself looked
# degenerate at some budget -- worth seeing next to the numbers rather than
# buried in the log above.
worst = {b: linear[b].get("sanity_severity", "ok") for b in sorted(linear)}
flagged = {b: s for b, s in worst.items() if s != "ok"}
if flagged:
    print(f"sanity: {flagged}  <- check the run log for details")
else:
    print("sanity: ok at every budget")

In [ ]:
# Package the results as ONE zip at the top of /kaggle/working, then delete the
# loose files -- the same shape both extraction notebooks and
# run_al_baseline.ipynb use.
#
# Kaggle's Output tab lists what is left in /kaggle/working when the session
# ends, and in a "Save & Run All" session that is the ONLY way to get a file
# out: there is no terminal and no kaggle CLI. Keeping the originals beside the
# zip also doubles the download, and a session over the ~20 GB Output quota
# shows NOTHING at all, including the files that were fine.
import shutil

from utils import results_archive_stem

SOURCE = Path(OUTPUT_DIR)
WORKING = Path("/kaggle/working")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
# make_archive must not write inside the directory being archived, or it packs
# a partial copy of itself on a second run.
assert SOURCE.resolve() != WORKING.resolve(), (
    "OUTPUT_DIR must be a subdirectory of /kaggle/working, not /kaggle/working itself"
)

STEM = results_archive_stem(DATASET, SAMPLER, SEED)
ARCHIVE = WORKING / STEM
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6

print(f"{ARCHIVE.name}.zip  ({size_mb:.1f} MB) contains:")
for path in sorted(SOURCE.rglob("*")):
    if path.is_file():
        print(f"    {path.relative_to(SOURCE)}  ({path.stat().st_size / 1e6:.2f} MB)")

shutil.rmtree(SOURCE, ignore_errors=True)

remaining = sorted(p for p in WORKING.iterdir() if p.name != "codapath")
total_mb = sum(
    f.stat().st_size for p in remaining for f in ([p] if p.is_file() else p.rglob("*"))
    if f.is_file()
) / 1e6
print(f"\n/kaggle/working now holds {total_mb:.1f} MB (Output quota ~20 GB):")
for path in remaining:
    print(f"    {path.name}{'/' if path.is_dir() else ''}")

print(f"""
NEXT STEPS (no terminal needed)
  1. Output tab (right panel) -> download {ARCHIVE.name}.zip
  2. kaggle.com/datasets -> New Dataset -> upload that zip
  3. In evaluate_al_sampler.ipynb: Add Data -> your new dataset, then point
     CHECKPOINT_ROOT at it to rebuild the table and fit PALM.""")